# Gu–Kelly–Xiu (2020) — simplified classroom pipeline

**Paper:** Gu, S., Kelly, B., & Xiu, D. (2020). *Empirical Asset Pricing via Machine Learning.* *Review of Financial Studies*, 33(5), 2223–2273.

This notebook is a **pedagogical sketch**, not a replication. We use a **synthetic stock–month panel** with a planted sparse nonlinear signal so that:

1. the pipeline runs in minutes with no CRSP/Compustat license;
2. students still practice **temporal splits**, a **model ladder**, **OOS R²**, and **long–short evaluation**.

Magnitudes will **not** match the paper. The design choices should.

**On the day:** notebook `01` first (real FF + lagged ML), then this one (`01a`).

**Companion:** `sessions/01a_gu_kelly_xiu.md`

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2020)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1. Synthetic panel (stand-in for characteristics → next-month return)

We generate \(N\) stocks × \(T\) months and \(P\) characteristics.

True conditional expected return (unknown to the models):

$$
\mathbb{E}[r_{i,t+1} \mid x_{i,t}]
= \beta_1 x_{i,t,1} + \beta_2 x_{i,t,2}^2 + \beta_3 \, x_{i,t,3} x_{i,t,4}
$$

plus noise. Signal is **stronger than in real markets** so the classroom ladder is readable in a few minutes; still noisy enough to discuss SNR.

In [ ]:
N, T, P = 200, 180, 12  # stocks, months, characteristics
try:
    dates = pd.date_range("2005-01-31", periods=T, freq="ME")
except ValueError:
    dates = pd.date_range("2005-01-31", periods=T, freq="M")
stocks = np.arange(N)

rows = []
for t, dt in enumerate(dates):
    X = rng.normal(size=(N, P))
    # cross-sectional rank transform ≈ common AP practice
    X_rank = (pd.DataFrame(X).rank(axis=0, pct=True).to_numpy() - 0.5) * 2

    f = (
        0.02 * X_rank[:, 0]
        + 0.015 * (X_rank[:, 1] ** 2)
        + 0.025 * (X_rank[:, 2] * X_rank[:, 3])
    )
    eps = rng.normal(scale=0.05, size=N)
    r = f + eps

    df_t = pd.DataFrame(X_rank, columns=[f"x{j}" for j in range(P)])
    df_t["stock"] = stocks
    df_t["date"] = dt
    df_t["ret"] = r
    df_t["f_true"] = f
    rows.append(df_t)

panel = pd.concat(rows, ignore_index=True)
features = [c for c in panel.columns if c.startswith("x")]
panel.head()

In [ ]:
print(panel.shape)
print("date range:", panel["date"].min().date(), "→", panel["date"].max().date())
panel["ret"].describe().round(4)

## 2. Temporal train / validation / test

**Never** shuffle stock–months randomly for this problem.

Here: fixed calendar split (expanding / rolling schemes are a good *do differently* exercise).

In [ ]:
train_end = pd.Timestamp("2014-12-31")
val_end = pd.Timestamp("2016-12-31")

train = panel[panel["date"] <= train_end]
val = panel[(panel["date"] > train_end) & (panel["date"] <= val_end)]
test = panel[panel["date"] > val_end]

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {df['date'].nunique():3d} months | {len(df):6d} rows")

X_train, y_train = train[features].to_numpy(), train["ret"].to_numpy()
X_val, y_val = val[features].to_numpy(), val["ret"].to_numpy()
X_test, y_test = test[features].to_numpy(), test["ret"].to_numpy()

## 3. OOS \(R^2\) (predictive)

Following the spirit of Gu–Kelly–Xiu, predictive OOS \(R^2\) compares squared forecast errors to squared returns (benchmark forecast of 0):

$$
R^2_{\mathrm{OOS}}
= 1 - \frac{\sum (r_{i,t+1} - \hat r_{i,t+1})^2}{\sum r_{i,t+1}^2}
$$

Tiny positive values can still be economically meaningful when turned into portfolios.

In [ ]:
def r2_oos(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum(y_true ** 2)
    return 1.0 - ss_res / ss_tot


def eval_split(model, X, y, label):
    pred = model.predict(X)
    return {"split": label, "r2_oos": r2_oos(y, pred), "pred": pred}

## 4. Model ladder

Classroom subset of the paper’s ladder:

| Model | Intent |
|-------|--------|
| OLS | Unregularized linear baseline |
| Ridge | Shrinkage (high-P discipline) |
| Random Forest | Nonlinearities / interactions |
| Small MLP | Flexible approximation |

Hyperparameters: light touch on **validation** only (not test).

In [ ]:
candidates = {
    "OLS": LinearRegression(),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ]),
    "RF": RandomForestRegressor(
        n_estimators=200,
        max_depth=4,
        min_samples_leaf=50,
        random_state=2020,
        n_jobs=-1,
    ),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            alpha=1e-3,
            max_iter=500,
            random_state=2020,
        )),
    ]),
}

fitted = {}
val_scores = []
for name, model in candidates.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    s = eval_split(model, X_val, y_val, "val")
    val_scores.append({"model": name, "r2_oos_val": s["r2_oos"]})
    print(f"{name:5s}  val R2_oos = {s['r2_oos']:.4f}")

val_leaderboard = pd.DataFrame(val_scores).sort_values("r2_oos_val", ascending=False)
val_leaderboard

## 5. Test-set comparison + long–short portfolios

Each month: sort stocks by \(\hat r\), long top decile / short bottom decile (equal weight).

In [ ]:
def long_short_returns(df, pred, q=0.1):
    """Month-by-month long top-q / short bottom-q portfolio returns."""
    out = df[["date", "ret"]].copy()
    out["pred"] = pred
    port = []
    for dt, g in out.groupby("date"):
        lo = g["pred"].quantile(q)
        hi = g["pred"].quantile(1 - q)
        long = g.loc[g["pred"] >= hi, "ret"].mean()
        short = g.loc[g["pred"] <= lo, "ret"].mean()
        port.append({"date": dt, "ls": long - short})
    return pd.DataFrame(port).set_index("date").sort_index()


def sharpe(monthly_returns):
    mu = monthly_returns.mean()
    sd = monthly_returns.std(ddof=1)
    return np.sqrt(12) * mu / sd if sd > 0 else np.nan


rows = []
ls_paths = {}
for name, model in fitted.items():
    pred = model.predict(X_test)
    r2 = r2_oos(y_test, pred)
    ls = long_short_returns(test, pred)
    ls_paths[name] = ls["ls"]
    rows.append({
        "model": name,
        "r2_oos_test": r2,
        "ls_mean_monthly": ls["ls"].mean(),
        "ls_sharpe_ann": sharpe(ls["ls"]),
    })

# Oracle: predict with true conditional mean (upper bound for this DGP)
oracle_pred = test["f_true"].to_numpy()
ls_oracle = long_short_returns(test, oracle_pred)
rows.append({
    "model": "ORACLE_f",
    "r2_oos_test": r2_oos(y_test, oracle_pred),
    "ls_mean_monthly": ls_oracle["ls"].mean(),
    "ls_sharpe_ann": sharpe(ls_oracle["ls"]),
})
ls_paths["ORACLE_f"] = ls_oracle["ls"]

results = pd.DataFrame(rows).sort_values("r2_oos_test", ascending=False)
results.round(4)

In [ ]:
cum = pd.DataFrame({k: (1 + v).cumprod() for k, v in ls_paths.items()})
ax = cum.plot(title="Cumulative long–short (test period)")
ax.set_ylabel("Growth of $1")
plt.tight_layout()
plt.show()

## 6. Discussion questions (on the podium)

1. Which model wins on **\(R^2_{\mathrm{OOS}}\)** vs **Sharpe**? Do they agree?
2. Why can OLS lose when the DGP has **squares and interactions**?
3. What breaks if we had used **random** `train_test_split`?
4. How would **transaction costs** change the ranking?
5. Map each step back to Gu–Kelly–Xiu: what did we oversimplify?

## 7. Experiments — "do differently"

Uncomment / extend one cell. Keep test metrics honest (tune only on val).

In [ ]:
# Experiment A: drop nonlinear-relevant features x1–x3 and re-fit Ridge / RF
# Experiment B: rolling train window (e.g. last 60 months only)
# Experiment C: long-short with q=0.2 instead of 0.1
# Experiment D: add a simple turnover penalty to monthly LS returns
# Experiment E: more compute / depth — 2-layer MLP vs 5 hidden layers

q_alt = 0.2
name = val_leaderboard.iloc[0]["model"]
pred = fitted[name].predict(X_test)
ls_alt = long_short_returns(test, pred, q=q_alt)
print(f"Best-on-val model={name}, q={q_alt}")
print("mean monthly LS:", round(ls_alt["ls"].mean(), 4))
print("ann. Sharpe:    ", round(sharpe(ls_alt["ls"]), 4))

# E) Does a deeper net win? Same split, same loss; only architecture changes.
depth_grid = {
    "MLP_2_(32,16)": (32, 16),
    "MLP_5_(32,32,32,32,32)": (32, 32, 32, 32, 32),
    "MLP_5_(64,64,64,64,64)": (64, 64, 64, 64, 64),
}
depth_rows = []
for dname, hid in depth_grid.items():
    m = Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            hidden_layer_sizes=hid, activation="relu", alpha=1e-3,
            max_iter=500, random_state=2020,
        )),
    ])
    m.fit(X_train, y_train)
    nn = m.named_steps["model"]
    p_val, p_test = m.predict(X_val), m.predict(X_test)
    ls = long_short_returns(test, p_test)
    n_params = sum(c.size for c in nn.coefs_) + sum(b.size for b in nn.intercepts_)
    depth_rows.append({
        "model": dname,
        "n_params": n_params,
        "r2_oos_val": r2_oos(y_val, p_val),
        "r2_oos_test": r2_oos(y_test, p_test),
        "ls_sharpe_ann": sharpe(ls["ls"]),
    })
pd.DataFrame(depth_rows).round(4)

## Takeaway

Gu–Kelly–Xiu is not "which neural net wins." It is a **research design** for ML in the cross-section: characteristics, disciplined OOS evaluation, a model ladder, and economic portfolio metrics under low SNR.

**Next (optional Module A+):** Chen–Pelger–Zhu (2024) — what changes when the objective encodes no-arbitrage rather than pure forecast MSE?